In [7]:
# 讀取資料
train_data = pd.read_csv('/kaggle/input/datasets/ccl499/fist-train/train.csv')
test_data = pd.read_csv('/kaggle/input/datasets/ccl499/first-test/test.csv')

# 看看考古題 (train_data) 的前五行長什麼樣子
train_data.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
# 看看哪些欄位有缺漏、有多少「詭異」的空白
train_data.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [10]:
# 看看年齡欄位的統計數字（平均值、最大最小值等）
train_data['Age'].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

In [11]:
# 把缺漏的年齡填入中位數 28
train_data['Age'] = train_data['Age'].fillna(28)

# 填完後，我們再檢查一次看看還有沒有缺漏
train_data['Age'].isnull().sum()

np.int64(0)

In [12]:
# 看看哪個港口最熱門
print(train_data['Embarked'].value_counts())

# 大家都從 'S' 上船，我們就把缺的那兩個填 'S'
train_data['Embarked'] = train_data['Embarked'].fillna('S')

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64


In [14]:
# 把沒救的欄位直接拿掉
train_data = train_data.drop(columns=['Cabin'])

KeyError: "['Cabin'] not found in axis"

In [17]:
# 1. 補齊 Embarked 的那 2 個洞
train_data['Embarked'] = train_data['Embarked'].fillna('S')

# 2. 進行「語言翻譯」：把文字變數字
# 性別：男=0, 女=1
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})

# 港口：S=0, C=1, Q=2
train_data['Embarked'] = train_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 3. 最後確認：看看現在的資料是不是變得很「數學」了
print("目前缺漏值數量：")
print(train_data.isnull().sum())
print("\n前五行資料長相：")
train_data.head(6)

目前缺漏值數量：
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex            891
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         0
dtype: int64

前五行資料長相：


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",NaN,22.0,1,0,A/5 21171,7.2500,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",NaN,38.0,1,0,PC 17599,71.2833,0
2,3,1,3,"Heikkinen, Miss. Laina",NaN,26.0,0,0,STON/O2. 3101282,7.9250,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",NaN,35.0,1,0,113803,53.1000,0
4,5,0,3,"Allen, Mr. William Henry",NaN,35.0,0,0,373450,8.0500,0
5,6,0,3,"Moran, Mr. James",NaN,28.0,0,0,330877,8.4583,0


In [18]:
from sklearn.ensemble import RandomForestClassifier

# 1. 定義「正確答案」：Survived 這一欄
y = train_data["Survived"]

# 2. 定義「考卷題目」：我們選出的 5 個重點特徵
features = ["Pclass", "Sex", "Age", "SibSp", "Parch"]
X = train_data[features]

# 3. 準備好隨機森林模型（設定 100 棵決策樹）
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)

# 4. 開始訓練！讓 AI 去讀考古題（X）與正確答案（y）
model.fit(X, y)

print("模型訓練完成！AI 已經學會如何預測生存率了。")

模型訓練完成！AI 已經學會如何預測生存率了。


In [19]:
# 叫 AI 吐出它心中最重要的因素排行榜
importances = pd.DataFrame({'特徵': features, '重要程度': model.feature_importances_})
importances = importances.sort_values(by='重要程度', ascending=False)

print("AI 判定生存關鍵因素排行榜：")
print(importances)

AI 判定生存關鍵因素排行榜：
       特徵      重要程度
0  Pclass  0.437644
2     Age  0.309046
3   SibSp  0.147336
4   Parch  0.105974
1     Sex  0.000000


In [21]:
# 1. 處理正式考卷 (test_data) 的數據，必須跟 train_data 完全一樣
test_data['Age'] = test_data['Age'].fillna(28)
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median()) # 考卷裡 Fare 多缺一筆，順手補上
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})
test_data['Embarked'] = test_data['Embarked'].fillna('S')
test_data['Embarked'] = test_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 2. 拿出考卷題目
X_test = test_data[features]

# 3. 讓 AI 開始作答（預測）
predictions = model.predict(X_test)

# 4. 把答案整理成 Kaggle 要的格式
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})

# 5. 存成檔案準備上傳！
output.to_csv('submission.csv', index=False)

print("恭喜！預測檔案 'submission.csv' 已經打造完成了！")

恭喜！預測檔案 'submission.csv' 已經打造完成了！


In [25]:
# 1. 重新讀取原始資料（把被玩壞的資料重置）
train_data = pd.read_csv('/kaggle/input/datasets/ccl499/fist-train/train.csv')

# 2. 重新處理資料（這次只按一次喔！）
train_data['Age'] = train_data['Age'].fillna(28)
train_data['Embarked'] = train_data['Embarked'].fillna('S')
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})
train_data['Embarked'] = train_data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 3. 重新訓練模型
y = train_data["Survived"]
features = ["Pclass", "Sex", "Age", "SibSp", "Parch"]
X = train_data[features]
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)

# 4. 再次檢查排行榜
importances = pd.DataFrame({'特徵': features, '重要程度': model.feature_importances_})
print(importances.sort_values(by='重要程度', ascending=False))

       特徵      重要程度
1     Sex  0.529516
0  Pclass  0.217894
2     Age  0.149220
3   SibSp  0.064152
4   Parch  0.039218


In [26]:
import os  #呼叫路徑指令
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ccl499/fist-train/train.csv
/kaggle/input/datasets/ccl499/first-test/test.csv
